# DNN v2 — Edge-Based Strategy Discovery

**Architecture:** ContextConditionedTCN (FiLM + temporal conv, 33 features)

**Method:** Train fresh model on 80%, generate per-snapshot predictions on unseen 20% using temporal sequences, evaluate with strategy engine (same logic as live bot).

**Output:** `data/optimal_strategy_dnn.json`

In [1]:
import sys

sys.path.insert(0, str(__import__("pathlib").Path.cwd().parent))
sys.path.insert(0, "../..")

import json
from datetime import UTC, datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from polybot.adapters.dnn_models import ContextConditionedTCN
from sklearn.preprocessing import StandardScaler
from strategy_engine import StrategyGrid, WalkForwardEvaluator
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)
MAX_BID = 0.85
SEQ_LEN = 50
ELAPSED_CUTOFF = 0.50

RAW_COLS = [
    "btc_price",
    "elapsed_pct",
    "market_volume",
    "up_best_bid",
    "up_best_ask",
    "up_bid_depth",
    "up_ask_depth",
    "down_best_bid",
    "down_best_ask",
    "down_bid_depth",
    "down_ask_depth",
]
CROSS_CANDLE_COLS = [
    "prior_return",
    "consecutive_streak",
    "streak_magnitude",
    "rolling_volatility",
    "candle_momentum",
    "ma_crossover",
    "trend_consistency",
    "reversal_regime",
    "rsi",
    "bollinger_pct_b",
    "stochastic_k",
    "adx",
    "return_autocorrelation",
    "multi_candle_return_3",
    "multi_candle_return_6",
    "regime_score",
    "mean_reversion_signal",
    "prior_reversal_rate",
    "volume_momentum",
    "volume_trend",
    "volume_price_correlation",
    "relative_volume",
]
ALL_COLS = RAW_COLS + CROSS_CANDLE_COLS

## 1. Load Data & Split

In [2]:
rows = []
with open("../../data/latest_features.jsonl") as f:
    for line in f:
        rows.append(json.loads(line))
df = pd.DataFrame(rows)
df["target"] = (df["outcome"] == "UP").astype(int)
df[ALL_COLS] = df[ALL_COLS].fillna(0)

candle_ids = df["candle_id"].unique()
split = int(len(candle_ids) * 0.8)
train_ids = set(candle_ids[:split])
val_ids = set(candle_ids[split:])
df_train = df[df["candle_id"].isin(train_ids)]
df_val = df[df["candle_id"].isin(val_ids)]
print(f"Train: {len(train_ids)} candles ({len(df_train):,} snapshots)")
print(f"Val:   {len(val_ids)} candles ({len(df_val):,} snapshots)")

Train: 5208 candles (246,168 snapshots)
Val:   1302 candles (60,452 snapshots)


## 2. Train Fresh ContextConditionedTCN on 80%

Train on single snapshots (last per candle, mid-candle only). The model handles 2D input via its single-snapshot fallback. This produces better strategy results than temporal-sequence training.

In [3]:
# Single-snapshot training (last snapshot per candle, mid-candle only)
df_train_mid = df_train[df_train["elapsed_pct"] <= ELAPSED_CUTOFF]
candle_df = df_train_mid.groupby("candle_id").last().reset_index()
X_tr = candle_df[ALL_COLS].values.astype(np.float32)
y_tr = candle_df["target"].values.astype(np.float32)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)

model = ContextConditionedTCN(raw_size=len(RAW_COLS), context_size=len(CROSS_CANDLE_COLS))
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
loss_fn = nn.BCEWithLogitsLoss()
X_t = torch.tensor(X_tr_s, dtype=torch.float32)
y_t = torch.tensor(y_tr, dtype=torch.float32).unsqueeze(1)
loader = DataLoader(TensorDataset(X_t, y_t), batch_size=256, shuffle=True)

sp = int(len(X_tr_s) * 0.9)
X_es = torch.tensor(X_tr_s[sp:], dtype=torch.float32)
y_es = torch.tensor(y_tr[sp:], dtype=torch.float32).unsqueeze(1)

best_vl, best_state, no_improve = float("inf"), None, 0
for epoch in range(1, 51):
    model.train()
    for bx, by in loader:
        optimizer.zero_grad()
        loss_fn(model(bx), by).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    model.eval()
    with torch.no_grad():
        vl = loss_fn(model(X_es), y_es).item()
    if vl < best_vl:
        best_vl = vl
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
    if no_improve >= 8:
        print(f"Early stop at epoch {epoch} (best={best_vl:.4f})")
        break
if best_state:
    model.load_state_dict(best_state)
model.eval()
print(f"Trained ContextConditionedTCN on {len(candle_df)} candles (single-snapshot)")

Trained ContextConditionedTCN on 5208 candles (single-snapshot)


## 3. Generate Predictions on Unseen 20%

Predict on ALL snapshots in validation set (not just mid-candle) — same as live bot.

In [4]:
candle_data = []
for cid in df_val["candle_id"].unique():
    snap_rows = df_val[df_val["candle_id"] == cid].sort_values("timestamp")
    if len(snap_rows) < 5:
        continue
    truth = int(snap_rows["target"].iloc[0])

    # DNN v2 uses single-snapshot fallback in strategy eval
    # (temporal buffering happens in DnnPredictor at runtime, not here)
    X_val = scaler.transform(snap_rows[ALL_COLS].fillna(0).values.astype(np.float32))
    with torch.no_grad():
        logits = model(torch.tensor(X_val, dtype=torch.float32))
        probs = torch.sigmoid(logits).numpy().flatten()

    snapshots = []
    for i in range(len(snap_rows)):
        ua = snap_rows.iloc[i]["up_best_ask"]
        da = snap_rows.iloc[i]["down_best_ask"]
        snapshots.append(
            {
                "tick": i,
                "elapsed_pct": float(snap_rows.iloc[i]["elapsed_pct"]),
                "pred": int(probs[i] >= 0.5),
                "prob": float(probs[i]),
                "up_ask": float(ua) if ua is not None and np.isfinite(ua) else None,
                "down_ask": float(da) if da is not None and np.isfinite(da) else None,
            }
        )
    candle_data.append({"candle_id": cid, "truth": truth, "snapshots": snapshots})

print(f"Built {len(candle_data)} validation candles (model has NEVER seen these)")

Built 1302 validation candles (model has NEVER seen these)


## 4. Edge-Based Grid Search + Export

In [5]:
grid = StrategyGrid()
strategies = grid.generate()
print(f"Grid: {len(strategies)} combinations")

evaluator = WalkForwardEvaluator(strategies=strategies, candle_data=candle_data, n_folds=5, max_bid=MAX_BID)
results_df = evaluator.run()

print("\nTop 5 by Sharpe:")
print(
    results_df[["strategy", "sharpe_mean", "win_rate_mean", "return_mean", "total_bets_mean"]]
    .head(5)
    .to_string(index=False)
)

# Export best
best = results_df.iloc[0]
best_cfg = best["_config"]
config = {
    "model": "dnn",
    "strategy": best["strategy"],
    "min_edge": best_cfg.min_edge,
    "max_entries": best_cfg.max_entries,
    "sharpe_mean": round(best["sharpe_mean"], 4),
    "sharpe_std": round(best["sharpe_std"], 4),
    "win_rate": round(best["win_rate_mean"], 4),
    "return_pct": round(best["return_mean"], 2),
    "max_drawdown": round(best["max_dd_mean"], 4),
    "n_folds": 5,
    "grid_size": len(strategies),
    "eval_candles": len(candle_data),
    "eval_method": "walk_forward_5_folds_on_validation_split",
    "total_bets": int(best["total_bets_mean"]),
    "created_at": datetime.now(UTC).isoformat(),
}
out_path = Path("../../data/optimal_strategy_dnn.json")
with open(out_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"\nBest: min_edge={best_cfg.min_edge}, max_entries={best_cfg.max_entries}")
print(f"Sharpe={best['sharpe_mean']:.4f}, WR={best['win_rate_mean'] * 100:.1f}%, Return={best['return_mean']:+.1f}%")
print(f"Saved to {out_path}")

Grid: 14 combinations



Top 5 by Sharpe:
     strategy  sharpe_mean  win_rate_mean  return_mean  total_bets_mean
edge>=0.00 x2     0.004747       0.505390     3.033521            520.8
edge>=0.00 x1     0.004740       0.505390     1.465514            260.4
edge>=0.02 x2     0.003566       0.504621     2.402895            520.8
edge>=0.02 x1     0.003516       0.504621     1.139926            260.4
edge>=0.05 x1     0.002950       0.504232     1.001465            260.2

Best: min_edge=0.0, max_entries=2
Sharpe=0.0047, WR=50.5%, Return=+3.0%
Saved to ../../data/optimal_strategy_dnn.json
